In [1]:
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

In [2]:
PROJECT_DIR = Path.cwd()

INPUT_PATH = PROJECT_DIR / "data" / "trades.json"
OUTPUT_PATH = PROJECT_DIR / "outputs" / "parsed_trades.json"

print(INPUT_PATH)
print(OUTPUT_PATH)

/Users/zhangmeng/Desktop/6822/group/data/trades.json
/Users/zhangmeng/Desktop/6822/group/outputs/parsed_trades.json


In [3]:
with open("data/trades.json", "r", encoding="utf-8") as f:
    base_trades = json.load(f)

with open("data/custom_trades.json", "r", encoding="utf-8") as f:
    custom_trades = json.load(f)

trades = base_trades + custom_trades

print("Total trades:", len(trades))

print(type(trades))
print(len(trades))
print(trades[0])

Total trades: 33
<class 'list'>
33
{'trade_id': 'T001', 'asset_class': 'Rates', 'instrument_type': 'Swap', 'use_case': 'Fixed_Float', 'reporting_counterparty_lei': '5493001KJTIIGC8Y1R12', 'other_counterparty_lei': '2138002TXD6KSZ3V5X27', 'uti': '5493001KJTIIGC8Y1R1220241219TRD00001', 'upi': None, 'execution_timestamp': '2024-12-19T09:35:00Z', 'effective_date': '2024-12-21', 'maturity_date': '2029-12-21', 'notional_currency': 'USD', 'notional_amount': 50000000, 'fixed_rate': 0.04825, 'reference_rate': 'USD-SOFR-COMPOUND', 'reference_rate_term_value': 3, 'reference_rate_term_unit': 'MNTH', 'payment_frequency': 'Quarterly', 'cleared': True, 'clearing_house': 'CME', 'platform': 'SEF', 'action_type': 'NEW', 'collateral_portfolio_code': 'PORT-A123', 'initial_margin_posted': 1250000, 'variation_margin_posted': 0}


In [4]:
CONVENTIONAL_ASSET_CLASSES = {
    "Rates",
    "Credit",
    "FX",
    "Equity",
    "Commodities"
}

NOVEL_ASSET_CLASSES = {
    "EventContract",
    "PredictionMarket",
    "PredictionContract"
}

ISO_UTC_TIMESTAMP_REGEX = re.compile(
    r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$"
)

DATE_REGEX = re.compile(
    r"^\d{4}-\d{2}-\d{2}$"
)

In [5]:
def safe_get(trade: Dict[str, Any], key: str, default=None):
    return trade.get(key, default)


def is_missing(value: Any) -> bool:
    return value is None or value == "" or value == []


def validate_timestamp(value: Optional[str]) -> Optional[str]:
    if is_missing(value):
        return "MISSING_EXECUTION_TIMESTAMP"

    if not isinstance(value, str):
        return "INVALID_EXECUTION_TIMESTAMP_TYPE"

    if not ISO_UTC_TIMESTAMP_REGEX.match(value):
        return "INVALID_EXECUTION_TIMESTAMP_FORMAT"

    return None


def validate_date(value: Optional[str], field_name: str) -> Optional[str]:
    if is_missing(value):
        return f"MISSING_{field_name.upper()}"

    if not isinstance(value, str):
        return f"INVALID_{field_name.upper()}_TYPE"

    if not DATE_REGEX.match(value):
        return f"INVALID_{field_name.upper()}_FORMAT"

    return None

In [6]:
def classify_instrument(trade: Dict[str, Any]) -> str:
    asset_class = safe_get(trade, "asset_class")
    instrument_type = safe_get(trade, "instrument_type")

    if is_missing(asset_class) or is_missing(instrument_type):
        return "CLASSIFICATION_AMBIGUOUS"

    if asset_class in CONVENTIONAL_ASSET_CLASSES:
        return "CONVENTIONAL_DERIVATIVE"

    if asset_class in NOVEL_ASSET_CLASSES:
        return "NOVEL_INSTRUMENT_NO_TAXONOMY"

    return "CLASSIFICATION_AMBIGUOUS"


def determine_parse_status(errors: List[str]) -> str:
    if len(errors) == 0:
        return "SUCCESS"

    fatal_errors = {
        "MISSING_TRADE_ID",
        "MISSING_ASSET_CLASS",
        "MISSING_INSTRUMENT_TYPE"
    }

    if any(error in fatal_errors for error in errors):
        return "FAILED"

    return "PARTIAL"

In [7]:
def normalize_upper(value):
    if isinstance(value, str):
        return value.strip().upper()
    return value


def normalize_string(value):
    if isinstance(value, str):
        return value.strip()
    return value


def normalize_boolean(value):
    if isinstance(value, bool):
        return value
    
    if isinstance(value, str):
        value_clean = value.strip().lower()
        if value_clean in ["true", "yes", "y", "1"]:
            return True
        if value_clean in ["false", "no", "n", "0"]:
            return False
    
    return value


def normalize_trade(trade: Dict[str, Any]) -> Dict[str, Any]:
    normalized = trade.copy()

    currency_fields = [
        "notional_currency",
        "settlement_currency",
        "notional_currency_leg1",
        "notional_currency_leg2"
    ]

    uppercase_fields = [
        "action_type",
        "platform",
        "clearing_house",
        "delivery_type",
        "option_type",
        "reference_rate",
        "reference_rate_leg1",
        "reference_rate_leg2",
        "underlying_currency_pair"
    ]

    boolean_fields = [
        "cleared"
    ]

    for field in currency_fields:
        if field in normalized:
            normalized[field] = normalize_upper(normalized[field])

    for field in uppercase_fields:
        if field in normalized:
            normalized[field] = normalize_upper(normalized[field])

    for field in boolean_fields:
        if field in normalized:
            normalized[field] = normalize_boolean(normalized[field])

    return normalized

In [8]:
def parse_trade(trade: Dict[str, Any]) -> Dict[str, Any]:
    raw_trade = trade.copy()
    trade = normalize_trade(trade)
    errors: List[str] = []

    trade_id = safe_get(trade, "trade_id")
    asset_class = safe_get(trade, "asset_class")
    instrument_type = safe_get(trade, "instrument_type")
    use_case = safe_get(trade, "use_case")

    if is_missing(trade_id):
        errors.append("MISSING_TRADE_ID")

    if is_missing(asset_class):
        errors.append("MISSING_ASSET_CLASS")

    if is_missing(instrument_type):
        errors.append("MISSING_INSTRUMENT_TYPE")

    if is_missing(use_case):
        errors.append("MISSING_USE_CASE")

    timestamp_error = validate_timestamp(
        safe_get(trade, "execution_timestamp")
    )
    if timestamp_error:
        errors.append(timestamp_error)

    classification_flag = classify_instrument(trade)

    if classification_flag == "CONVENTIONAL_DERIVATIVE":
        effective_date_error = validate_date(
            safe_get(trade, "effective_date"),
            "effective_date"
        )
        if effective_date_error:
            errors.append(effective_date_error)
    
    elif classification_flag == "NOVEL_INSTRUMENT_NO_TAXONOMY":
        settlement_date_error = validate_date(
            safe_get(trade, "settlement_date"),
            "settlement_date"
        )
        if settlement_date_error:
            errors.append(settlement_date_error)

    maturity_date = safe_get(trade, "maturity_date")
    expiry_date = safe_get(trade, "expiry_date")
    settlement_date = safe_get(trade, "settlement_date")

    if is_missing(maturity_date) and is_missing(expiry_date) and is_missing(settlement_date):
        errors.append("MISSING_MATURITY_EXPIRY_OR_SETTLEMENT_DATE")
    else:
        if not is_missing(maturity_date):
            maturity_error = validate_date(maturity_date, "maturity_date")
            if maturity_error:
                errors.append(maturity_error)

        if not is_missing(expiry_date):
            expiry_error = validate_date(expiry_date, "expiry_date")
            if expiry_error:
                errors.append(expiry_error)

        if not is_missing(settlement_date):
            settlement_error = validate_date(settlement_date, "settlement_date")
            if settlement_error:
                errors.append(settlement_error)
                
    classified_fields = {
        "notional_currency": safe_get(trade, "notional_currency"),
        "notional_amount": safe_get(trade, "notional_amount"),
        "cleared": safe_get(trade, "cleared"),
        "clearing_house": safe_get(trade, "clearing_house"),
        "platform": safe_get(trade, "platform"),
        "action_type": safe_get(trade, "action_type"),
        "uti": safe_get(trade, "uti"),
        "upi": safe_get(trade, "upi"),
        "execution_timestamp": safe_get(trade, "execution_timestamp"),
        "effective_date": safe_get(trade, "effective_date"),
        "maturity_date": safe_get(trade, "maturity_date"),
        "expiry_date": safe_get(trade, "expiry_date"),
        "settlement_date": safe_get(trade, "settlement_date"),
        "reporting_counterparty_lei": safe_get(trade, "reporting_counterparty_lei"),
        "other_counterparty_lei": safe_get(trade, "other_counterparty_lei"),
    }

    return {
        "trade_id": trade_id,
        "parse_status": determine_parse_status(errors),
        "asset_class": asset_class,
        "instrument_type": instrument_type,
        "use_case": use_case,
        "classification_flag": classification_flag,
        "parse_errors": errors,
        "classified_fields": classified_fields,
        "normalized_trade": trade,
        "raw_trade": raw_trade
    }

In [9]:
parsed_trades = []

for trade in trades:
    if not isinstance(trade, dict):
        parsed_trades.append({
            "trade_id": None,
            "parse_status": "FAILED",
            "asset_class": None,
            "instrument_type": None,
            "use_case": None,
            "classification_flag": "CLASSIFICATION_AMBIGUOUS",
            "parse_errors": ["TRADE_RECORD_NOT_OBJECT"],
            "classified_fields": {},
            "normalized_trade": {},
            "raw_trade": trade
        })
    else:
        parsed_trades.append(parse_trade(trade))

OUTPUT_PATH = PROJECT_DIR / "outputs" / "parsed_trades.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(parsed_trades, f, indent=2, ensure_ascii=False)

print(f"Saved updated parsed trades to: {OUTPUT_PATH}")
print(f"Total trades parsed: {len(parsed_trades)}")

Saved updated parsed trades to: /Users/zhangmeng/Desktop/6822/group/outputs/parsed_trades.json
Total trades parsed: 33


In [10]:
for record in parsed_trades:
    if record["trade_id"] in ["T026", "T027", "T028"]:
        print(record["trade_id"], record["classification_flag"], record["parse_status"], record["parse_errors"])

T026 NOVEL_INSTRUMENT_NO_TAXONOMY SUCCESS []
T027 NOVEL_INSTRUMENT_NO_TAXONOMY SUCCESS []
T028 NOVEL_INSTRUMENT_NO_TAXONOMY SUCCESS []


In [11]:
for record in parsed_trades:
    if record["parse_errors"]:
        print(record["trade_id"], record["parse_status"], record["parse_errors"])

T013 PARTIAL ['INVALID_EXECUTION_TIMESTAMP_FORMAT']


In [12]:
MODULE1_OUTPUT_PATH = PROJECT_DIR / "outputs" / "module1_parser_output.json"

module1_outputs = []

for record in parsed_trades:
    module1_outputs.append({
        "trade_id": record["trade_id"],
        "parse_status": record["parse_status"],
        "asset_class": record["asset_class"],
        "instrument_type": record["instrument_type"],
        "use_case": record["use_case"],
        "classification_flag": record["classification_flag"],
        "parse_errors": record["parse_errors"]
    })

with open(MODULE1_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(module1_outputs, f, indent=2, ensure_ascii=False)

print(f"Saved Module 1 output to: {MODULE1_OUTPUT_PATH}")

Saved Module 1 output to: /Users/zhangmeng/Desktop/6822/group/outputs/module1_parser_output.json


In [13]:
import pandas as pd

summary_df = pd.DataFrame([
    {
        "trade_id": r["trade_id"],
        "parse_status": r["parse_status"],
        "asset_class": r["asset_class"],
        "instrument_type": r["instrument_type"],
        "use_case": r["use_case"],
        "classification_flag": r["classification_flag"],
        "parse_errors": "; ".join(r["parse_errors"])
    }
    for r in parsed_trades
])

summary_df

,trade_id,parse_status,asset_class,instrument_type,use_case,classification_flag,parse_errors
0,T001,SUCCESS,Rates,Swap,Fixed_Float,CONVENTIONAL_DERIVATIVE,
1,T002,SUCCESS,Credit,Swap,Corporate,CONVENTIONAL_DERIVATIVE,
2,T003,SUCCESS,FX,Forward,NDF,CONVENTIONAL_DERIVATIVE,
3,T004,SUCCESS,Equity,Option,SingleName_Put,CONVENTIONAL_DERIVATIVE,
4,T005,SUCCESS,Rates,Swap,Fixed_Float,CONVENTIONAL_DERIVATIVE,
5,T006,SUCCESS,Rates,Swap,Basis,CONVENTIONAL_DERIVATIVE,
6,T007,SUCCESS,Commodities,Swap,SingleName,CONVENTIONAL_DERIVATIVE,
7,T008,SUCCESS,Rates,Option,Swaption,CONVENTIONAL_DERIVATIVE,
8,T009,SUCCESS,FX,Option,Vanilla,CONVENTIONAL_DERIVATIVE,
9,T010,SUCCESS,Credit,Swap,Index,CONVENTIONAL_DERIVATIVE,


In [14]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(parsed_trades, f, indent=2, ensure_ascii=False)

print(f"Saved parsed trades to: {OUTPUT_PATH}")

Saved parsed trades to: /Users/zhangmeng/Desktop/6822/group/outputs/parsed_trades.json


In [15]:
summary_path = PROJECT_DIR / "outputs" / "parser_summary.csv"

summary_df.to_csv(summary_path, index=False)

print(f"Saved summary CSV to: {summary_path}")

Saved summary CSV to: /Users/zhangmeng/Desktop/6822/group/outputs/parser_summary.csv


In [16]:
custom_ids = ["C001", "C002", "C003", "C004", "C005"]

for trade in trades:
    if trade["trade_id"] in custom_ids:
        print("\n", trade["trade_id"])
        print("asset_class:", trade.get("asset_class"))
        print("notional_currency:", trade.get("notional_currency"))
        print("other_counterparty_lei:", trade.get("other_counterparty_lei"))
        print("uti:", trade.get("uti"))


 C001
asset_class: Rates
notional_currency: USD
other_counterparty_lei: 2138002TXD6KSZ3V5X27
uti: 5493001KJTIIGC8Y1R1220260501IRS0001

 C002
asset_class: Credit
notional_currency: EUR
other_counterparty_lei: BAD_LEI
uti: VGRQXHF3J8VDLUA7XE9220260502CDS0001

 C003
asset_class: FX
notional_currency: USD
other_counterparty_lei: 4R3ZURLYISNNNMHMK608
uti: 1VUV7VQFKUOQSJ21A20820260503FXF0001

 C004
asset_class: Equity
notional_currency: INVALID_CCY
other_counterparty_lei: 2138002TXD6KSZ3V5X27
uti: 5493001KJTIIGC8Y1R1220260504EQO0001

 C005
asset_class: EventContract
notional_currency: USDC
other_counterparty_lei: None
uti: None


In [17]:
for record in parsed_trades:
    if record["trade_id"] in custom_ids:
        print(
            record["trade_id"],
            record["parse_status"],
            record["asset_class"],
            record["classification_flag"],
            record["parse_errors"]
        )

C001 SUCCESS Rates CONVENTIONAL_DERIVATIVE []
C002 SUCCESS Credit CONVENTIONAL_DERIVATIVE []
C003 SUCCESS FX CONVENTIONAL_DERIVATIVE []
C004 SUCCESS Equity CONVENTIONAL_DERIVATIVE []
C005 SUCCESS EventContract NOVEL_INSTRUMENT_NO_TAXONOMY []


In [18]:
custom_error_summary = [
    {
        "trade_id": "C002",
        "intentional_error": "Invalid other_counterparty_lei",
        "error_value": "BAD_LEI",
        "expected_engine_detection": "INVALID_LEI_OTHER / NON_COMPLIANT"
    },
    {
        "trade_id": "C004",
        "intentional_error": "Invalid notional_currency",
        "error_value": "INVALID_CCY",
        "expected_engine_detection": "INVALID_CURRENCY / INVALID_ATTRIBUTES"
    }
]

custom_error_summary

[{'trade_id': 'C002',
  'intentional_error': 'Invalid other_counterparty_lei',
  'error_value': 'BAD_LEI',
  'expected_engine_detection': 'INVALID_LEI_OTHER / NON_COMPLIANT'},
 {'trade_id': 'C004',
  'intentional_error': 'Invalid notional_currency',
  'error_value': 'INVALID_CCY',
  'expected_engine_detection': 'INVALID_CURRENCY / INVALID_ATTRIBUTES'}]